# Model output metrics

Tables and figures for the 2-stage XGBoost model and the three reference
baselines, at ADM1 level, which is where impact is reported.

Run order: coverage first. The metrics below are only comparable across models
when they are scored on the same administrative units.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.evaluation.bootstrap import bootstrap_binary_metrics
from visualization.lib import (CATEGORY_LABELS, MODEL_LABELS,
                                            MODEL_ORDER, STRATEGIES,
                                            absolute_errors, categorise,
                                            coverage, load_all, load_variants,
                                            plot_categorical_confusion_matrices,
                                            plot_models_violin,
                                            PAPER_MODEL_NAMES,
                                            restrict_to_common,
                                            attach_event_time,
                                            rolling_metrics_with_bootstrap,
                                            plot_rolling_comparison,
                                            plot_cumulative_events,
                                            RUN_SUFFIX)

STRATEGY = "global"   # global | walk_forward | geo_constrained
LEVEL = "adm1"
EVENT_COL = "sid"     # resample cyclones, not country-event records
# Both layouts carry these; the aliases the training path adds are
# absent from the rule bundles.
TARGET_COL = "reported"
PRED_COL = "predicted"
N_BOOT = 1000

# Figures are written here as well as shown, so the repository carries
# the current ones. A run read under a suffix writes its own folder,
# so the figures of one run never land on another's.
FIGDIR = Path.cwd() / f"figs{RUN_SUFFIX}"
FIGDIR.mkdir(exist_ok=True)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

models = load_all(strategy=STRATEGY, level=LEVEL)
print(f"loaded: {sorted(models)}")
if not models:
    raise SystemExit(f"no outputs for strategy {STRATEGY!r} at level {LEVEL!r}")

## 1. Coverage

`shared_units` is the number of (event, ADM1) pairs every loaded model scored.
Where a model scores fewer, the comparison is restricted to the shared set
before any metric is computed.

In [ ]:
cov = coverage(models)
display(cov)

if len(models) > 1 and "only_this_model" in cov and cov["only_this_model"].sum():
    print("models differ in coverage; restricting to shared units")
    models = restrict_to_common(models)
    display(coverage(models))

## 2. Binary metrics with cyclone-clustered confidence intervals

Two thresholds: any reported impact, and impact above 15% of an ADM1 unit's
population. Intervals come from resampling cyclones, so events of the same
storm stay together.

In [ ]:
rows = []
for name in MODEL_ORDER:
    if name not in models:
        continue
    m = bootstrap_binary_metrics(
        models[name], event_col=EVENT_COL, target_col=TARGET_COL, pred_col=PRED_COL, thresholds=(0, 15),
        model_name=MODEL_LABELS.get(name, name), n_boot=N_BOOT, random_state=42)
    rows.append(m)

metrics = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
point_cols = [c for c in ["model", "threshold", "precision", "recall",
                          "specificity", "accuracy", "f1", "csi"]
              if c in metrics.columns]
display(metrics[point_cols].round(4))

In [ ]:
def with_ci(df, metric, decimals=3):
    lo, hi = f"{metric}_ci_low", f"{metric}_ci_high"
    if lo not in df.columns:
        return df[metric].round(decimals).astype(str)
    return (df[metric].round(decimals).astype(str) + " ("
            + df[lo].round(decimals).astype(str) + ", "
            + df[hi].round(decimals).astype(str) + ")")

if not metrics.empty:
    table = pd.DataFrame({"model": metrics["model"], "threshold": metrics["threshold"]})
    for m in ["precision", "recall", "f1", "csi", "accuracy"]:
        if m in metrics.columns:
            table[m] = with_ci(metrics, m)
    display(table)

## 3. Categorical confusion matrices

Rows are reported, columns predicted, in the three impact categories. Cells are
row-normalised, so each row shows how units of that reported class were
classified.

In [ ]:
present = [n for n in MODEL_ORDER if n in models]
dfs = [models[n] for n in present]
names = [PAPER_MODEL_NAMES[MODEL_ORDER.index(n)] for n in present]

fig, matrices = plot_categorical_confusion_matrices(
    dfs, names, savefig=FIGDIR / "categorical_confusion_matrices.png")
plt.show()

print("matrix totals:", [int(m.sum()) for m in matrices])
print("all equal:", len({int(m.sum()) for m in matrices}) == 1)

## 4. Absolute error

Left: the full distribution. Right: the same data restricted to the lower
range, where most units sit.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 4), sharex=True, sharey=False,
                         gridspec_kw={"height_ratios": [2, 1]})

plot_models_violin(axes[0], dfs, names, show_legend=True,
                   show_ylabel=False, show_info=True)
plot_models_violin(axes[1], dfs, names, show_legend=False,
                   show_ylabel=False, show_info=True)
axes[1].set_ylim(0, 15)          # the bulk of the distribution
fig.text(-0.01, 0.5, "Absolute error (percentage points)", va="center", ha="center",
         rotation="vertical", fontsize=12)
plt.tight_layout()
plt.savefig(FIGDIR / "absolute_error_violins.png", dpi=300,
            bbox_inches="tight", pad_inches=0.5)
plt.show()

summary = pd.DataFrame({
    "model": names,
    "MAE": [float(np.abs(d["predicted"] - d["reported"]).mean()) for d in dfs],
    "median AE": [float(np.abs(d["predicted"] - d["reported"]).median()) for d in dfs],
    "p90 AE": [float(np.abs(d["predicted"] - d["reported"]).quantile(0.9)) for d in dfs],
}).round(3)
display(summary)

## 5. Cross-validation strategies

The same model under each strategy. `walk_forward` trains only on cyclones
preceding the test cyclone, so the earliest events in the record cannot be
scored and its event count is lower by construction.

In [ ]:
rows = []
for strategy in STRATEGIES:
    got = load_all(strategy=strategy, level=LEVEL, models=["2-stage-XGBoost"])
    if not got:
        print(f"{strategy}: not scored yet")
        continue
    df = got["2-stage-XGBoost"]
    m = bootstrap_binary_metrics(
        df, event_col=EVENT_COL, target_col=TARGET_COL, pred_col=PRED_COL, thresholds=(0, 15),
        model_name=strategy, n_boot=N_BOOT, random_state=42)
    m["events"] = df["DisNo."].nunique()
    m["units"] = len(df)
    rows.append(m)

if rows:
    by_strategy = pd.concat(rows, ignore_index=True)
    cols = [c for c in ["model", "threshold", "events", "units", "precision",
                        "recall", "f1", "csi"] if c in by_strategy.columns]
    display(by_strategy[cols].round(4))

In [ ]:
if rows:
    fig, ax = plt.subplots(figsize=(7, 4))
    width = 0.35
    for k, thr in enumerate((0, 15)):
        sub = by_strategy[by_strategy["threshold"] == thr]
        x = np.arange(len(sub))
        lo = (sub["f1"] - sub["f1_ci_low"]).to_numpy() if "f1_ci_low" in sub else None
        hi = (sub["f1_ci_high"] - sub["f1"]).to_numpy() if "f1_ci_high" in sub else None
        ax.bar(x + k * width, sub["f1"], width,
               yerr=None if lo is None else [lo, hi], capsize=3,
               label=f"threshold {thr}%")
        ax.set_xticks(x + width / 2, sub["model"], rotation=15)
    ax.set_ylabel("F1 (ADM1)")
    ax.set_title("F1 by cross-validation strategy")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIGDIR / "f1_by_strategy.png", dpi=300,
                bbox_inches="tight")
    plt.show()

### Categorical confusion matrices per strategy

The same model under each cross-validation strategy, so the effect of restricting the training set is visible per category rather than only in a summary score.

In [ ]:
strategy_frames, strategy_names = [], []
for strategy in STRATEGIES:
    got = load_all(strategy=strategy, level=LEVEL, models=["2-stage-XGBoost"])
    if got:
        strategy_frames.append(got["2-stage-XGBoost"])
        strategy_names.append(strategy.replace("_", "-"))

if strategy_frames:
    # Restricting to the units every strategy scored: walk-forward cannot
    # score the earliest events, so the panels would otherwise be built from
    # different rows.
    shared = restrict_to_common(dict(zip(strategy_names, strategy_frames)))
    dfs_s = [shared[n] for n in strategy_names]
    print(f"rows per strategy after restricting to the shared units: "
          f"{[len(d) for d in dfs_s]}")

    fig, mats = plot_categorical_confusion_matrices(
        dfs_s, strategy_names,
        savefig=FIGDIR / "strategy_confusion_matrices.png")
    plt.show()

    rows = []
    for name, d, m in zip(strategy_names, dfs_s, mats):
        rows.append({"strategy": name,
                     "high caught": int(m[0, 2]),
                     "high missed as low": int(m[0, 1]),
                     "high missed as none": int(m[0, 0]),
                     "no-impact called high": int(m[2, 2]),
                     "MAE": float((d["predicted"] - d["reported"]).abs().mean())})
    display(pd.DataFrame(rows).round(3))

## 6. Variants of the model

Three models of the same administrative units: one fitted per grid cell and aggregated afterwards, one fitted on the units directly, and one fitted without the count of previous events. All use the same strategy and the same tuned parameters, so the difference is the model.

In [ ]:
variants = load_variants()
print(f"scored: {sorted(variants)}")

for label, d in variants.items():
    print(f"  {label:<18} rows {len(d):>7,}  events {d['DisNo.'].nunique():>4}"
          f"  ADM1 {d['GID_1'].nunique():>5}")

if len(variants) > 1:
    variants = restrict_to_common(variants)
    print(f"\nrestricted to the units every variant scored: "
          f"{len(next(iter(variants.values()))):,}")

In [ ]:
if len(variants) > 1:
    rows = []
    for label, d in variants.items():
        if EVENT_COL not in d.columns:
            d = d.merge(
                pd.concat([v[["DisNo.", EVENT_COL]] for v in variants.values()
                           if EVENT_COL in v.columns]).drop_duplicates(),
                on="DisNo.", how="left")
        rows.append(bootstrap_binary_metrics(
            d, event_col=EVENT_COL, thresholds=(0, 15),
            target_col=TARGET_COL, pred_col=PRED_COL,
            model_name=label, n_boot=N_BOOT, random_state=42))

    variant_metrics = pd.concat(rows, ignore_index=True).sort_values(
        ["threshold", "model"]).reset_index(drop=True)

    table = variant_metrics[["model", "threshold"]].copy()
    for m in ["precision", "recall", "f1", "csi"]:
        table[m] = with_ci(variant_metrics, m)
    table["MAE"] = with_ci(variant_metrics, "mean_abs_error", 2)
    for t in (0, 15):
        print(f"\n=== threshold {t}% ===")
        display(table[table["threshold"] == t].drop(columns="threshold")
                .reset_index(drop=True))

In [ ]:
if len(variants) > 1:
    labels = list(variants)
    fig, mats = plot_categorical_confusion_matrices(
        [variants[l] for l in labels], labels,
        savefig=FIGDIR / "model_variants_confusion_matrices.png")
    plt.show()

## 7. Walk-forward against leave-one-out, over time

Walk-forward trains only on cyclones preceding the one it predicts, so early in the record it has little to learn from. Comparing the two over a trailing five-year window shows how much of the gap is the method and how much is the size of the record available at the time.

In [ ]:
rolling = {}
for name, strategy in (("walk", "walk_forward"), ("loocv", "global")):
    got = load_all(strategy=strategy, level=LEVEL, models=["2-stage-XGBoost"])
    if got:
        rolling[name] = attach_event_time(got["2-stage-XGBoost"])

if len(rolling) == 2:
    print(f"walk-forward {len(rolling['walk']):,} rows, "
          f"LOOCV {len(rolling['loocv']):,} rows")
    walk, loocv = {}, {}
    for t in (0, 15):
        walk[t] = rolling_metrics_with_bootstrap(
            rolling["walk"], event_col=EVENT_COL, threshold=t,
            resolution="month", window_size=5, n_boot=30, seed=42)
        loocv[t] = rolling_metrics_with_bootstrap(
            rolling["loocv"], event_col=EVENT_COL, threshold=t,
            resolution="month", window_size=5, n_boot=30, seed=42)
        print(f"  threshold {t}%: {len(walk[t])} windows")
else:
    print("both strategies are needed for this comparison")

In [ ]:
if len(rolling) == 2:
    fig = plot_rolling_comparison(
        walk, loocv, from_date="2005-01-01",
        savefig=FIGDIR / "walk_forward_vs_loocv_rolling.png")
    plt.show()

In [ ]:
if "loocv" in rolling:
    fig, cumulative = plot_cumulative_events(
        rolling["loocv"], from_year=2000,
        savefig=FIGDIR / "cumulative_training_events.png")
    plt.show()
    print(f"cyclone-events available by {int(cumulative.index.max())}: "
          f"{int(cumulative.iloc[-1]):,}")